# ChromaDB Basic Operations - Practice Notebook


**Assumed pre-installed libraries:** `chromadb`, `sentence-transformers`

This notebook teaches the core operations of a vector database (Chroma)
using a small, real dataset: factual statements about ancient Indian
contributions to mathematics, astronomy, medicine, metallurgy, and
linguistics. You'll create a collection, add documents with metadata,
run semantic searches, update and delete entries, filter by metadata, and
see a hands-on demonstration of *why* vector databases exist at all.


## 1. Why Vector Databases Are Needed

A brute-force search — comparing a query embedding against every single
stored vector — works fine for a handful of documents, but becomes
impractically slow once a knowledge base grows to hundreds of thousands or
millions of chunks. A vector database exists to solve two problems at that
scale:

- **Fast approximate nearest-neighbor (ANN) search** — specialized indexing
  structures let the system find the top-k most similar vectors in far less
  time than a full brute-force comparison, trading a small amount of
  accuracy for a large gain in speed.
- **Persistence and management** — storing vectors alongside their
  metadata, supporting inserts, updates, and deletes over time, and
  filtering search results by metadata (for example, restricting a search
  to only chunks tagged with a particular category or era).

Our dataset here is small (a few dozen statements), so brute force would be
plenty fast. We're using it to learn the *operations* clearly; Part 8 of
this notebook comes back to actually measuring the brute-force-vs-index
speed difference at a larger, synthetic scale, so the motivation above isn't
just a claim you have to take on faith.


## 2. Indexing Concepts (Conceptual Overview)

- **Flat / brute-force index** — no cleverness: compares the query against
  every stored vector. Returns exact results, but query time scales
  linearly with the number of stored vectors, making it slow at scale.
  Useful as a ground-truth baseline to measure approximate indexes against.
- **IVF (Inverted File Index)** — clusters all vectors into a fixed number
  of buckets (using an algorithm like k-means) when the index is built. At
  query time, only the handful of buckets closest to the query are
  searched, rather than the entire dataset. A parameter called `nprobe`
  controls how many buckets are checked: checking more buckets improves
  accuracy (recall) at the cost of higher query latency.
- **HNSW (Hierarchical Navigable Small World)** — builds a multi-layer graph
  in which each vector is connected to its approximate nearest neighbors. A
  search "navigates" this graph, starting from a coarse top layer and
  descending to a fine-grained bottom layer near the query. HNSW generally
  achieves very high recall with fast query times, at the cost of higher
  memory usage and slower index-build time compared to IVF.

**Where Chroma fits in:** Chroma uses HNSW internally as its default index
for approximate nearest-neighbor search -- you don't have to choose between
flat/IVF/HNSW yourself the way you would with a lower-level library like
FAISS (see Week 2, Day 4). What you *can* configure are HNSW's own tuning
parameters, via the collection's `metadata` -- for example `hnsw:space`
(the distance metric), `hnsw:construction_ef`, and `hnsw:M` (which trade
index-build time and memory for recall, the same trade-off described
above). We'll set some of these explicitly below so the concept isn't just
abstract.


## 3. The Dataset: Bhartiya Ancient Science & Mathematics

Each entry is a short factual statement, tagged with a `category` (the
broad field), a `scholar_or_text` (who or what source it's associated
with), and an approximate `era`. This metadata is what lets us filter
search results later (Part 7), not just search by meaning.


In [ ]:
bhartiya_science_facts = [
    {
        "id": "fact_01",
        "text": "Aryabhata calculated the value of pi to four decimal places "
                "(3.1416) in his 5th-century work, the Aryabhatiya.",
        "category": "mathematics",
        "scholar_or_text": "Aryabhata / Aryabhatiya",
        "era": "5th century CE",
    },
    {
        "id": "fact_02",
        "text": "The Baudhayana Sulba Sutra, used for Vedic altar construction, "
                "contains an early statement of what is now called the "
                "Pythagorean theorem, predating Pythagoras.",
        "category": "mathematics",
        "scholar_or_text": "Baudhayana Sulba Sutra",
        "era": "circa 800 BCE",
    },
    {
        "id": "fact_03",
        "text": "Brahmagupta was among the first mathematicians to give "
                "systematic rules for arithmetic using zero as a number, "
                "in his 7th-century text the Brahmasphutasiddhanta.",
        "category": "mathematics",
        "scholar_or_text": "Brahmagupta / Brahmasphutasiddhanta",
        "era": "7th century CE",
    },
    {
        "id": "fact_04",
        "text": "The Bakhshali manuscript contains one of the earliest known "
                "uses of a zero placeholder symbol in a mathematical text "
                "from the Indian subcontinent.",
        "category": "mathematics",
        "scholar_or_text": "Bakhshali manuscript",
        "era": "early centuries CE (exact dating debated)",
    },
    {
        "id": "fact_05",
        "text": "The decimal place-value number system that spread from India "
                "to the Islamic world and then to Europe forms the basis of "
                "the number system used worldwide today.",
        "category": "mathematics",
        "scholar_or_text": "Indian decimal system",
        "era": "1st millennium CE",
    },
    {
        "id": "fact_06",
        "text": "Pingala's Chandahshastra describes a binary-like classification "
                "system for poetic meters, using short and long syllables, "
                "centuries before binary numbers were formalized in the West.",
        "category": "mathematics",
        "scholar_or_text": "Pingala / Chandahshastra",
        "era": "circa 3rd-2nd century BCE",
    },
    {
        "id": "fact_07",
        "text": "Bhaskara II's Bijaganita presents methods for solving "
                "quadratic and indeterminate equations, and his broader work "
                "is credited by some historians with early ideas resembling "
                "differential calculus, centuries before Newton and Leibniz.",
        "category": "mathematics",
        "scholar_or_text": "Bhaskara II / Bijaganita",
        "era": "12th century CE",
    },
    {
        "id": "fact_08",
        "text": "The Kerala school of astronomy and mathematics, founded by "
                "Madhava of Sangamagrama, developed infinite series expansions "
                "for trigonometric functions, including a series for pi now "
                "sometimes called the Madhava-Leibniz series, predating "
                "Leibniz by roughly two centuries.",
        "category": "mathematics",
        "scholar_or_text": "Madhava / Kerala School",
        "era": "14th-15th century CE",
    },
    {
        "id": "fact_09",
        "text": "Aryabhata proposed that the Earth rotates on its own axis, "
                "explaining the apparent daily motion of the stars, and gave "
                "a geometric explanation of lunar and solar eclipses using "
                "the shadows of the Earth and Moon.",
        "category": "astronomy",
        "scholar_or_text": "Aryabhata / Aryabhatiya",
        "era": "5th century CE",
    },
    {
        "id": "fact_10",
        "text": "Varahamihira's Pancha-Siddhantika compiled and compared five "
                "earlier astronomical treatises, showing evidence of exchange "
                "between Indian and Hellenistic astronomical traditions.",
        "category": "astronomy",
        "scholar_or_text": "Varahamihira / Pancha-Siddhantika",
        "era": "6th century CE",
    },
    {
        "id": "fact_11",
        "text": "Aryabhata estimated the length of the sidereal year to a value "
                "very close to the modern accepted figure, differing by only "
                "a small margin.",
        "category": "astronomy",
        "scholar_or_text": "Aryabhata / Aryabhatiya",
        "era": "5th century CE",
    },
    {
        "id": "fact_12",
        "text": "The Sushruta Samhita, an ancient Sanskrit surgical text, "
                "describes reconstructive nose surgery (rhinoplasty), leading "
                "some medical historians to call its author the father of "
                "plastic surgery.",
        "category": "medicine",
        "scholar_or_text": "Sushruta / Sushruta Samhita",
        "era": "ancient (exact dating debated)",
    },
    {
        "id": "fact_13",
        "text": "The Sushruta Samhita catalogs more than 300 surgical "
                "procedures and around 120 surgical instruments, including "
                "techniques for cataract surgery.",
        "category": "medicine",
        "scholar_or_text": "Sushruta / Sushruta Samhita",
        "era": "ancient (exact dating debated)",
    },
    {
        "id": "fact_14",
        "text": "The Charaka Samhita, a foundational text of Ayurveda, "
                "systematically discusses anatomy, physiology, and internal "
                "medicine, and is considered one of the earliest surviving "
                "texts on internal medicine.",
        "category": "medicine",
        "scholar_or_text": "Charaka / Charaka Samhita",
        "era": "ancient (compiled over centuries)",
    },
    {
        "id": "fact_15",
        "text": "The Iron Pillar of Delhi, dating to approximately the 4th "
                "century CE, has survived over 1,600 years with minimal "
                "rusting due to a protective passive oxide film, a subject of "
                "modern metallurgical research.",
        "category": "metallurgy",
        "scholar_or_text": "Iron Pillar of Delhi",
        "era": "circa 4th century CE",
    },
    {
        "id": "fact_16",
        "text": "Ancient Indian metallurgists produced wootz steel, a "
                "high-carbon crucible steel that was exported and later used "
                "to forge Damascus steel blades prized for their strength and "
                "distinctive surface patterns.",
        "category": "metallurgy",
        "scholar_or_text": "Wootz steel tradition",
        "era": "circa 1st millennium BCE onward",
    },
    {
        "id": "fact_17",
        "text": "Kanada, founder of the Vaisheshika school of philosophy, "
                "proposed that all matter is composed of indivisible "
                "particles called 'anu', an early atomic theory of matter.",
        "category": "philosophy_of_science",
        "scholar_or_text": "Kanada / Vaisheshika Sutra",
        "era": "circa 6th-2nd century BCE (dating debated)",
    },
    {
        "id": "fact_18",
        "text": "Panini's Ashtadhyayi is a highly systematic, rule-based "
                "grammar of Sanskrit; its formal structure has been compared "
                "by modern scholars to concepts in formal language theory and "
                "generative grammar.",
        "category": "linguistics",
        "scholar_or_text": "Panini / Ashtadhyayi",
        "era": "circa 5th-4th century BCE",
    },
]

print(f"Loaded {len(bhartiya_science_facts)} facts across "
      f"{len(set(f['category'] for f in bhartiya_science_facts))} categories.")


## 4. Creating a Chroma Client and Collection

Two client types worth knowing:
- `chromadb.Client()` — an in-memory (ephemeral) client; data is lost when
  the process ends. Good for experimentation, like this notebook.
- `chromadb.PersistentClient(path="./my_chroma_db")` — writes data to disk,
  so it survives across notebook sessions. This is what you'd use for a
  real project.

We'll also explicitly set two HNSW parameters on the collection
(`hnsw:space` and `hnsw:construction_ef`) to make the Part 2 discussion
concrete -- these directly correspond to the "index quality vs. build cost"
trade-off described there.


In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Using sentence-transformers explicitly (same model used in Week 2's
# notebooks) rather than Chroma's default, so we know exactly which
# embedding model is producing our vectors.
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.Client()   # in-memory for this notebook;
                              # use chromadb.PersistentClient(path=...) to persist to disk

collection = client.get_or_create_collection(
    name="bhartiya_science_facts",
    embedding_function=embedding_fn,
    metadata={
        "hnsw:space": "cosine",          # distance metric for the HNSW index
        "hnsw:construction_ef": 100,      # higher = better index quality, slower to build
    },
)

print("Collection created:", collection.name)


## 5. Adding Documents (Insert)

Chroma's `add` takes parallel lists: `documents` (the text to embed),
`ids` (a unique identifier per document), and `metadatas` (structured data
attached to each document, used for filtering later).


In [ ]:
collection.add(
    documents=[f["text"] for f in bhartiya_science_facts],
    ids=[f["id"] for f in bhartiya_science_facts],
    metadatas=[
        {"category": f["category"], "scholar_or_text": f["scholar_or_text"], "era": f["era"]}
        for f in bhartiya_science_facts
    ],
)

print("Documents in collection:", collection.count())


**Exercise 1:** Write 2 new facts of your own (any true, well-documented
fact about Indian ancient science, medicine, mathematics, or engineering --
double check it against a reliable source rather than guessing), give each
a unique `id`, `category`, `scholar_or_text`, and `era`, and add them to the
collection using `collection.add(...)`. Confirm `collection.count()`
increases by 2.


## 6. Querying (Semantic Search)

`collection.query` embeds the query text with the same embedding function
used at insert time, and returns the `n_results` most similar documents by
the configured distance metric.


In [ ]:
results = collection.query(
    query_texts=["What did ancient Indians know about the number zero?"],
    n_results=3,
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"distance={dist:.3f}  [{meta['category']}]  {doc}")


In [ ]:
# Try a query that's conceptually related but shares few exact words with
# the source text, to confirm this is genuine semantic search, not just
# keyword matching.
results = collection.query(
    query_texts=["Did ancient India know anything about surgery or treating the human body?"],
    n_results=3,
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"distance={dist:.3f}  [{meta['category']}]  {doc}")


**Exercise 2:** Write 3 of your own natural-language questions (not just
copies of the fact text) that you'd expect to match different categories in
the dataset (e.g., one about astronomy, one about metallurgy, one about
language/grammar). Run each through `collection.query` and check whether the
top result's `category` metadata matches what you expected.


## 7. Filtering by Metadata

Combine semantic search with a `where` filter to restrict results to a
specific category, scholar, or era -- useful when you know part of what
you're looking for structurally, not just semantically.


In [ ]:
# Semantic search restricted to only the 'astronomy' category
results = collection.query(
    query_texts=["calculations about the sky and planets"],
    n_results=3,
    where={"category": "astronomy"},
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"distance={dist:.3f}  [{meta['category']}]  {doc}")


In [ ]:
# get() retrieves by exact criteria, with no similarity ranking involved --
# useful for exact lookups rather than fuzzy search.
medicine_facts = collection.get(where={"category": "medicine"})
print(f"Found {len(medicine_facts['ids'])} medicine facts:")
for doc_id, doc in zip(medicine_facts["ids"], medicine_facts["documents"]):
    print(f"  [{doc_id}] {doc[:80]}...")


**Exercise 3 (mini deliverable):** Run a query filtered to `category:
mathematics` only, and separately a query filtered to `category: astronomy`
only, using the SAME query text ("what did ancient Indian scholars
calculate?"). Compare the top result in each case -- does restricting the
category change which fact is considered "most relevant," even though the
query itself didn't change?


## 8. Peek, Get by ID, Update, and Delete

Rounding out the basic operations: inspecting a sample of the collection,
fetching by exact ID, updating an existing entry in place, and deleting.


In [ ]:
# peek() returns a small sample of the collection, useful for a quick sanity check
sample = collection.peek(limit=2)
print("Sample IDs:", sample["ids"])


In [ ]:
# get() by exact id -- no embedding/similarity involved, a direct lookup
one_fact = collection.get(ids=["fact_01"])
print(one_fact["documents"][0])


In [ ]:
# update() modifies an existing document and/or its metadata in place,
# keeping the same id -- useful when a source fact needs a correction.
collection.update(
    ids=["fact_11"],
    documents=["Aryabhata's Aryabhatiya gives a sidereal year length that "
               "modern scholars note is remarkably close to the currently "
               "accepted astronomical value."],
)

updated = collection.get(ids=["fact_11"])
print(updated["documents"][0])


In [ ]:
# delete() removes documents by id (or by a `where` metadata filter)
print("Before delete:", collection.count())
collection.delete(ids=["fact_18"])
print("After delete:", collection.count())


**Exercise 4:** Delete every fact in the `philosophy_of_science` category
using `collection.delete(where=...)` instead of deleting by id. Confirm with
`collection.count()` and by re-running a query that previously would have
matched that category.


## 9. Empirical Demonstration: Why the Index Actually Matters at Scale

Our dataset has ~18-20 facts, so brute-force search is instant either way --
the "why vector databases are needed" argument from Part 1 isn't really
visible yet. Let's make it visible with a synthetic, larger set of random
vectors, comparing a manual brute-force numpy search against Chroma's
(HNSW-indexed) query time.

**A calibration note before you run this:** numpy's brute-force computation
is a single vectorized matrix multiply (BLAS-accelerated), so at a *moderate*
scale (tens of thousands of vectors) it can actually still be very fast --
comparable to or even faster than Chroma's query, once you include Chroma's
own per-call overhead. The linear-scaling argument from Part 1 is real, but
it only becomes clearly visible once `n_vectors` is large enough that
brute-force's O(n) cost genuinely dominates. The cell below uses a large
enough `n_vectors` for the effect to show up clearly; Exercise 5 has you
verify this scaling relationship yourself rather than take it on faith.


In [ ]:
import numpy as np
import time

rng = np.random.RandomState(42)
n_vectors = 200000
dim = 384   # matches all-MiniLM-L6-v2's embedding dimension

synthetic_vectors = rng.rand(n_vectors, dim).astype("float32")
synthetic_vectors /= np.linalg.norm(synthetic_vectors, axis=1, keepdims=True)
query_vector = rng.rand(1, dim).astype("float32")
query_vector /= np.linalg.norm(query_vector)

# --- Manual brute-force search (numpy) ---
start = time.perf_counter()
similarities = synthetic_vectors @ query_vector[0]   # cosine similarity (vectors are unit-normalized)
top_k_brute = np.argsort(-similarities)[:5]
brute_force_time = time.perf_counter() - start
print(f"Brute-force numpy search over {n_vectors} vectors: {brute_force_time*1000:.2f} ms")


In [ ]:
# --- Chroma (HNSW-indexed) search over the same synthetic vectors ---
synthetic_collection = client.get_or_create_collection(
    name="synthetic_benchmark",
    metadata={"hnsw:space": "cosine"},
)

# Chroma's add() can accept pre-computed embeddings directly (no text/embedding
# function needed), which is perfect for this synthetic, non-textual benchmark.
batch_size = 5000
for start_idx in range(0, n_vectors, batch_size):
    end_idx = min(start_idx + batch_size, n_vectors)
    synthetic_collection.add(
        ids=[f"vec_{i}" for i in range(start_idx, end_idx)],
        embeddings=synthetic_vectors[start_idx:end_idx].tolist(),
    )

start = time.perf_counter()
chroma_results = synthetic_collection.query(
    query_embeddings=query_vector.tolist(),
    n_results=5,
)
chroma_time = time.perf_counter() - start
print(f"Chroma (HNSW) search over {n_vectors} vectors: {chroma_time*1000:.2f} ms")

print(f"\nSpeedup: {brute_force_time / chroma_time:.1f}x" if chroma_time > 0 else "")


**Exercise 5 (mini deliverable):** Re-run Part 9 with `n_vectors = 20000`
(small), `n_vectors = 50000` (medium), and `n_vectors = 200000` (the
notebook's default, large) -- adjust `batch_size` if needed for memory.
Record the brute-force and Chroma query times at each size in a small
table. Does the *brute-force* time grow roughly linearly with `n_vectors`,
as the "Why Vector Databases Are Needed" section predicted? At what point
does Chroma's HNSW-indexed query time clearly pull ahead, and does that
match the calibration note above -- i.e., is the advantage small or even
absent at the smallest size you tried?

**Exercise 6 (discussion):** This benchmark measures *query* time only. What
did we *not* measure that the Day 4 material also flagged as a real cost of
HNSW indexes relative to a flat index (hint: re-read the HNSW bullet in Part
2)? Where in the code above would you expect to see that cost show up if you
timed it?


## 10. Summary

You've now performed every core Chroma operation -- create a client and
collection (with HNSW parameters configured explicitly), add documents with
metadata, run semantic search, filter by metadata, peek/get/update/delete,
and empirically confirmed why an ANN index like HNSW matters once a
collection grows large, using a real dataset of facts about ancient Indian
contributions to mathematics, astronomy, medicine, metallurgy, and
linguistics along the way.
